# Deploying NVIDIA Nemotron 3.5 Lightning with TensorRT-LLM

This notebook will walk you through how to run the NVIDIA Nemotron 3.5 Lightning NVFP4 checkpoint via TensorRT-LLM on a single H100.

[TensorRT-LLM](https://nvidia.github.io/TensorRT-LLM/) is NVIDIA's open-source library for accelerating and optimizing LLM inference performance on NVIDIA GPUs.

Nemotron 3.5 Lightning is published as two checkpoints:

- **BF16** — [`nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-BF16`](https://huggingface.co/nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-BF16)
- **NVFP4** — [`nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4`](https://huggingface.co/nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4) - **this notebook deploys this checkpoint**

**Model size:** 30B total parameters, 3B active (MoE)

Prerequisites for this notebook:
- 1x NVIDIA H100 80GB with recent drivers
- Python 3.10+
- Docker

## Overview

- **Serve** the Nemotron 3.5 Lightning NVFP4 checkpoint using TensorRT-LLM
- **Query the model** through an OpenAI-compatible API
- **Invoke tools** using structured function-calling outputs
- **Tune reasoning depth** by configuring the model's thinking budget

## Table of Contents

1. **Prerequisites & environment** - Launch Docker container
2. **Verify GPU** - Confirm CUDA and GPU availability
3. **OpenAI-compatible server** - Launch and query TensorRT-LLM
   - **Create a YAML file** - Base or MTP configuration
   - **Load the model** - NVFP4 on a single H100
   - **Generate responses** - Single, sequential, and streamed completions
   - **Reasoning** - Toggle thinking on/off
   - **Tool calling** - Function calling via OpenAI tools schema
   - **Controlling Reasoning Budget** - Limit reasoning trace length
4. **Cleanup and shutdown**

#### Launch on NVIDIA Brev
You can simplify the environment setup by using [NVIDIA Brev](https://developer.nvidia.com/brev). Click the button to launch the NVFP4 variant on a Brev instance with the necessary dependencies pre-configured.

Once deployed, click on the "Open Notebook" button to get started with this guide.

**For NVFP4 (1x H100):**

[![Launch on Brev](https://brev-assets.s3.us-west-1.amazonaws.com/nv-lb-dark.svg)](https://brev.nvidia.com/launchable/deploy?launchableID=env-3HawHhqQ43fhtSFmVFnqAtWCCGN)

## Prerequisites & environment

### Launch the Docker container

Open a terminal on the host and start an interactive shell inside the TensorRT-LLM container.

```shell
docker run --rm -it --ipc=host --ulimit memlock=-1 --ulimit stack=67108864 --gpus=all \
  --network=host \
  -v ~/.cache/huggingface:/root/.cache/huggingface \
  nvcr.io/nvidia/tensorrt-llm/release:1.3.0rc24
```

> **Note:** Mount the HuggingFace cache directory so model weights are read from disk rather than re-downloaded on each run. Replace `~/.cache/huggingface` if your cache is in a different location.

All `trtllm-serve` commands later in this notebook should be run from inside this container.

### Install notebook client dependencies

These are for the notebook only: `openai` sends the requests, `transformers` provides the tokenizer used in the reasoning budget section, and `torch` backs the GPU check. The container already carries everything the model needs in order to load and run.

> **Note:** Match the `torch` build to your driver. A CUDA 13 wheel on a CUDA 12 driver reports `CUDA available: False` even when the GPU is healthy. In case that happens, check the driver's CUDA version with `nvidia-smi` and install from the matching index if needed, for example `--index-url https://download.pytorch.org/whl/cu128`. This affects only the GPU check below, not the model serving.

In [1]:
#If pip not found
!python3 -m ensurepip --default-pip

Looking in links: /tmp/tmpfp6xpc7h
Processing /tmp/tmpfp6xpc7h/pip-25.0.1-py3-none-any.whl


In [ ]:
# Client-side dependencies.
%pip install openai==2.38.0 transformers==5.9.0 torch

## Verify GPU

Check that CUDA is available and the GPU is detected correctly.

> **Expected output:** `CUDA available: True` with one H100 listed. If CUDA is `False`, check your driver installation.

In [2]:
import sys
import torch

print(f"Python: {sys.version}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Num GPUs: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU[{i}]: {torch.cuda.get_device_name(i)}")

Python: 3.12.13 (main, Jul 23 2026, 14:43:28) [Clang 22.1.3 ]
CUDA available: True
Num GPUs: 1
GPU[0]: NVIDIA H100 PCIe


## OpenAI-compatible server

Start a local OpenAI-compatible server with TensorRT-LLM from inside the Docker container terminal.

### Create a YAML file with the required configuration

Run one of the following from the Docker terminal before serving. Both write the same file and configure the Marlin MoE backend with an FP8 KV cache and stochastic-rounded Mamba state, differing only in speculative decoding. The `trtllm-serve` command below is identical either way.

```shell
cat > ./extra-llm-api-config.yml << EOF
kv_cache_config:
  dtype: fp8
  enable_block_reuse: false
  free_gpu_memory_fraction: 0.8
  mamba_ssm_cache_dtype: float16
  mamba_ssm_stochastic_rounding: true
  mamba_ssm_philox_rounds: 5
  mamba_state_config:
    periodic_snapshot_interval: 8192

moe_config:
  backend: MARLIN

nvfp4_gemm_config:
  allowed_backends: [marlin, cutlass, cublaslt, cuda_core]

cuda_graph_config:
  enable_padding: true
  max_batch_size: 128

enable_chunked_prefill: true
num_postprocess_workers: 4
print_iter_log: true
stream_interval: 10
disable_overlap_scheduler: false
EOF
```

#### MTP

`speculative_config` turns on MTP (Multi-Token Prediction), where a small draft layer built into the model guesses ahead and the model verifies those guesses in one step. This checkpoint has a single MTP layer, applied repeatedly to produce the 3 draft tokens set by `max_draft_len`; the layer count comes from the checkpoint and is not configurable.

```shell
cat > ./extra-llm-api-config.yml << EOF
kv_cache_config:
  dtype: fp8
  enable_block_reuse: false
  free_gpu_memory_fraction: 0.8
  mamba_ssm_cache_dtype: float16
  mamba_ssm_stochastic_rounding: true
  mamba_ssm_philox_rounds: 5
  mamba_state_config:
    periodic_snapshot_interval: 8192

moe_config:
  backend: MARLIN

nvfp4_gemm_config:
  allowed_backends: [marlin, cutlass, cublaslt, cuda_core]

cuda_graph_config:
  enable_padding: true
  max_batch_size: 128

speculative_config:
  decoding_type: MTP
  max_draft_len: 3

enable_chunked_prefill: true
num_postprocess_workers: 4
print_iter_log: true
stream_interval: 10
disable_overlap_scheduler: false
EOF
```

### Configuration reference

| | NVFP4 |
|---|---|
| **Model** | `nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4` |
| **Hardware (this notebook)** | 1x H100 80GB |
| **Docker image** | `nvcr.io/nvidia/tensorrt-llm/release:1.3.0rc24` |
| **MoE backend** | `MARLIN` |
| **Mamba state cache** | FP16 with stochastic rounding |
| **Max batch size** | 128 |
| **Max num tokens** | 8192 |
| **Speculative decoding** | None or MTP (`max_draft_len: 3`) |
| **Reasoning parser** | `nemotron-v3` |
| **Tool parser** | `qwen3_coder` |
| **Port** | 8000 |

### Load the model

Run the following from inside the Docker terminal, in the same directory as `extra-llm-api-config.yml`.

> **Note:** Parser names are backend-specific and not interchangeable. TensorRT-LLM uses `--reasoning_parser nemotron-v3` and `--tool_parser qwen3_coder`. vLLM uses `--reasoning-parser nemotron_v3` and SGLang uses `--reasoning-parser nemotron_3` — these are different identifiers for the same logical capability.

```shell
trtllm-serve nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4 \
  --host 127.0.0.1 \
  --port 8000 \
  --max_batch_size 128 \
  --max_num_tokens 8192 \
  --trust_remote_code \
  --served_model_name nemotron-3.5-lightning \
  --reasoning_parser nemotron-v3 \
  --tool_parser qwen3_coder \
  --extra_llm_api_options extra-llm-api-config.yml
```

### Wait for the server to be ready

Poll `/v1/models` from a host terminal before sending any requests:

```shell
until curl -sf http://localhost:8000/v1/models > /dev/null 2>&1; do
  echo "Waiting for server..."; sleep 10
done
echo "Server is ready"
```

> **Expected output:** The loop prints `Waiting for server...` during model loading, then exits with `Server is ready` once the endpoint responds.

> **Note:** On the first run, the model weights will be downloaded from Hugging Face before loading begins, so the combined download and load time will be longer than on subsequent runs.

### Generate responses

The cells below show single, sequential, and streamed completions, followed by reasoning on/off, tool calling, and reasoning budget examples.

> **Note:** The reasoning trace and the final answer share one completion budget. If reasoning consumes all of `max_tokens`, the response comes back with `finish_reason: "length"` and `content` either empty or cut off mid-sentence. Raise `max_tokens` or shorten the prompt when that happens. The examples below budget a few thousand tokens for prompts that invite a long answer, and only a few hundred where reasoning is turned off.

> **Note:** `trtllm-serve` does not validate the `model` field on incoming requests, so a name that does not match the server is silently accepted rather than rejected.

In [2]:
from openai import OpenAI

# Set this to match the --served_model_name used when starting the server
SERVED_MODEL_NAME = "nemotron-3.5-lightning"
BASE_URL = "http://localhost:8000/v1"

client = OpenAI(base_url=BASE_URL, api_key="null")

In [5]:
# Single chat completion
response = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Give me 3 bullet points about TensorRT-LLM."},
    ],
    temperature=1.0,
    top_p=0.95,
    max_tokens=2048,
)
choice = response.choices[0]
print("Reasoning:", choice.message.reasoning_content)
print("Content:", choice.message.content)

Reasoning: Here, the user is asking for 3 bullet points about TensorRT-LLM. I need to provide concise, informative bullet points. I should recall what TensorRT-LLM is: it's a library for optimizing and deploying large language models (LLMs) on NVIDIA GPUs using TensorRT. Key points: performance optimization, ease of use, integration with frameworks, support for various model architectures, etc.

I'll structure three bullet points highlighting main features: high performance through kernel fusion and caching, ease of use with Python API and support for popular models, and compatibility with deployment on various NVIDIA platforms. I'll make sure each bullet is succinct and informative. Let's craft them.
Content: - **High-Performance Inference**: TensorRT-LLM accelerates large language models (LLMs) by leveraging NVIDIA TensorRT’s kernel optimization, operator fusion, and efficient memory management, delivering significantly higher throughput and lower latency compared to standard PyTorch

### Sequential completions

Send multiple prompts in sequence and collect all responses.

In [3]:
prompts = [
    "What is the square root of 144?",
    "What is the capital of France?",
    "Explain quantum computing in simple terms.",
]

for prompt in prompts:
    response = client.chat.completions.create(
        model=SERVED_MODEL_NAME,
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt},
        ],
        temperature=1.0,
        top_p=0.95,
        max_tokens=2048,
    )
    print(f"Q: {prompt}")
    print(f"A: {response.choices[0].message.content}\n")

Q: What is the square root of 144?
A: The square root of 144 is **12** (since 12 × 12 = 144). In mathematical terms, this is the principal (non-negative) square root.

Q: What is the capital of France?
A: The capital of France is **Paris**.

Q: Explain quantum computing in simple terms.
A: ### Computing in Two Worlds

**Classical computers** use **bits**. A bit is like a light switch: it’s either **off (0)** or **on (1)**. Every task-your phone does, a calculator solves, a video game runs-is built from millions of these binary switches flipping on and off in precise sequences.

**Quantum computers** use **qubits** (quantum bits). The key difference is that a qubit can be 0, 1, **or both at the same time**. This is called **superposition**.

---

### The "Spinning Coin" Analogy
Imagine a coin:
- **Classical bit:** A coin that’s either heads or tails. You look at it, and you know exactly what it is.
- **Qubit:** A coin spinning on a table. While it’s spinning, it’s effectively both heads

### Streamed generation

Receive tokens as they are generated using the OpenAI streaming API.

In [8]:
# Streaming chat completion
print("Streaming response:")
stream = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What are the first 5 prime numbers?"}
    ],
    temperature=1.0,
    top_p=0.95,
    max_tokens=1024,
    stream=True,
)

section = None

for chunk in stream:
    delta = chunk.choices[0].delta
    if not delta:
        continue

    reasoning = getattr(delta, "reasoning_content", None)
    if reasoning:
        if section != "reasoning":
            print("Reasoning: ", end="", flush=True)
            section = "reasoning"
        print(reasoning, end="", flush=True)

    if delta.content:
        if section != "content":
            print("\n\nContent: ", end="", flush=True)
            section = "content"
        print(delta.content, end="", flush=True)

Streaming response:
Reasoning: Here's a thinking process:

1.  **Analyze User Input:** User asks "What are the first 5 prime numbers?"
2.  **Identify Key Concept:** Prime numbers are natural numbers greater than 1 that have no positive divisors other than 1 and themselves.
3.  **Recall/Determine First 5 Primes:** 
   - 2 (smallest prime, only even prime)
   - 3
   - 5
   - 7
   - 11
   Let me verify: 
   - 2: prime ✓
   - 3: prime ✓
   - 4: not prime (divisible by 2)
   - 5: prime ✓
   - 6: not prime
   - 7: prime ✓
   - 8, 9, 10: not prime
   - 11: prime ✓
   So first 5 are 2, 3, 5, 7, 11.
4.  **Formulate Response:** List them clearly, maybe with a brief definition. Keep it concise.
5.  **Output Generation:** "The first 5 prime numbers are: 2, 3, 5, 7, and 11." Optionally add that prime numbers are greater than 1 with exactly two factors. I'll just give the direct answer.✅


Content: The first 5 prime numbers are: **2, 3, 5, 7, and 11**. 

(Prime numbers are natural numbers greater th

### Reasoning

The model supports two modes — **Reasoning ON** (default) and **Reasoning OFF**.

Toggle by setting `enable_thinking` to `False` in `chat_template_kwargs`. Use `temperature=1.0, top_p=0.95` when reasoning is on, and `temperature=0.2` when it is off.

In [9]:
# Reasoning on (default)
print("Reasoning on")
resp = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Write a haiku about GPUs."}
    ],
    temperature=1.0,
    top_p=0.95,
    max_tokens=4096,
)
print("Reasoning:", resp.choices[0].message.reasoning_content)
print("Content:", resp.choices[0].message.content)
print()

# Reasoning off
print("Reasoning off")
resp2 = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Give me 3 bullet points about TensorRT-LLM."}
    ],
    temperature=0.2,
    max_tokens=256,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}}
)
print("Content:", resp2.choices[0].message.content)

Reasoning on
Reasoning: Here thinking process:

1.  **Analyze the Request:**
   - User wants a haiku about GPUs.
   - Haiku structure: 5 syllables, 7 syllables, 5 syllables (total 17 syllables).
   - Topic: GPUs (Graphics Processing Units).

2.  **Understand GPUs:**
   - What are they? Processors for graphics, parallel computing, gaming, AI, rendering, screens, chips, cores, speed, parallel processing, visuals, etc.

3.  **Brainstorm Keywords & Syllable Counts:**
   - GPU: 1 syllable (G-P-U, but pronounced "G-P-U" or "jay-pee-you", usually 1 or 2? "GPU" is often said as "G-P-U" = 3 letters but 1 syllable if said as "jay-pee-you" it's 3 syllables, but as an acronym it's often 1. I'll avoid relying on "GPU" pronunciation. Better to use "graphics" or "chip".)
   - Let's list words with syllable counts:
     - graphics: 2 (graphics)
     - processor: 3 (pro-ces-sor)
     - chip: 1
     - core: 1
     - render: 2
     - parallel: 3
     - speed: 1
     - light: 1
     - screen: 1
     - pix

### Tool calling

Call functions using the OpenAI Tools schema and inspect the returned `tool_calls`.

In [10]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "calculate_tip",
            "parameters": {
                "type": "object",
                "properties": {
                    "bill_total": {
                        "type": "integer",
                        "description": "The total amount of the bill"
                    },
                    "tip_percentage": {
                        "type": "integer",
                        "description": "The percentage of tip to be applied"
                    }
                },
                "required": ["bill_total", "tip_percentage"]
            }
        }
    }
]

completion = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": ""},
        {"role": "user", "content": "My bill is $50. What will be the amount for 15% tip?"}
    ],
    tools=TOOLS,
    temperature=1.0,
    top_p=0.95,
    max_tokens=1024,
    stream=False
)

choice = completion.choices[0]
print("Reasoning:", choice.message.reasoning_content)
print("Tool calls:", choice.message.tool_calls)

Reasoning: Here's a thinking process:

1.  **Analyze User Input:**
   - Bill total: $50
   - Tip percentage: 15%
   - Question: What will be the amount for 15% tip?

2.  **Identify Required Tool:**
   - The `calculate_tip` function takes `bill_total` and `tip_percentage`.
   - `bill_total` should be 50
   - `tip_percentage` should be 15

3.  **Check Tool Parameters:**
   - `bill_type`: integer, description: "The total amount of the bill"
   - `tip_percentage`: integer, description: "The percentage of tip to be applied"
   - Both are required.

4.  **Prepare Function Call:**
   - `bill_total`: 50
   - `tip_percentage`: 15

5.  **Execute Function Call:**
   - I'll call `calculate_tip` with these parameters.

6.  **Verify Output/Format:**
   - The prompt says: "If you choose to call a function ONLY reply in the following format with NO suffix:"
   - I need to output the function call block correctly.

Let's do it.⟹

Tool calls: [ChatCompletionMessageFunctionToolCall(id='chatcmpl-tool-db53

### Controlling Reasoning Budget

The `reasoning_budget` parameter lets you limit how long the model reasons before producing a response. When the reasoning trace reaches the token budget, the model will try to wrap up at the next newline.

> **Note:** If no newline is encountered within 500 tokens after the budget threshold, the reasoning trace is forcibly terminated at `reasoning_budget + 500` tokens.

In [12]:
from typing import Any, Dict, List
import openai
from transformers import AutoTokenizer


class ThinkingBudgetClient:
    def __init__(self, base_url: str, api_key: str, tokenizer_name_or_path: str):
        self.base_url = base_url
        self.api_key = api_key
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_name_or_path)
        self.client = openai.OpenAI(base_url=self.base_url, api_key=self.api_key)

    def chat_completion(
        self,
        model: str,
        messages: List[Dict[str, Any]],
        reasoning_budget: int = 512,
        max_tokens: int = 1024,
        **kwargs,
    ) -> Dict[str, Any]:
        assert (
            max_tokens > reasoning_budget
        ), f"reasoning_budget must be smaller than max_tokens. Given {max_tokens=} and {reasoning_budget=}"

        response = self.client.chat.completions.create(
            model=model,
            messages=messages,
            max_tokens=reasoning_budget,
            **kwargs
        )

        reasoning_content = response.choices[0].message.reasoning_content or ""

        if "</think>" not in reasoning_content:
            reasoning_content = f"{reasoning_content}.\n</think>\n\n"

        reasoning_tokens_used = len(
            self.tokenizer.encode(reasoning_content, add_special_tokens=False)
        )
        remaining_tokens = max_tokens - reasoning_tokens_used

        assert (
            remaining_tokens > 0
        ), f"remaining tokens must be positive. Given {remaining_tokens=}. Increase max_tokens or lower reasoning_budget."

        messages.append({"role": "assistant", "content": reasoning_content})
        prompt = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            continue_final_message=True,
        )

        response = self.client.completions.create(
            model=model,
            prompt=prompt,
            max_tokens=remaining_tokens,
            **kwargs
        )

        return {
            "reasoning_content": reasoning_content.strip().strip("</think>").strip(),
            "content": response.choices[0].text,
            "finish_reason": response.choices[0].finish_reason,
        }

In [13]:
budget_client = ThinkingBudgetClient(
    base_url="http://localhost:8000/v1",
    api_key="null",
    tokenizer_name_or_path="nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4"  # use actual HF model ID for tokenizer
)

In [14]:
resp = budget_client.chat_completion(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Write a haiku about GPUs."}
    ],
    temperature=1.0,
    max_tokens=1024,
    reasoning_budget=128
)
print("Reasoning:", resp["reasoning_content"])
print("Content:", resp["content"])

Reasoning: Here's a thinking process:

1.  **Analyze the Request**: The user wants a haiku about GPUs.
2.  **Understand the Format**: A haiku is a traditional Japanese poem with a 5-7-5 syllable structure (total 17 syllables), typically focusing on nature or a moment in time, but can be about any subject.
3.  **Identify the Subject**: GPUs (Graphics Processing Units). Key themes: graphics, computing, parallel processing, gaming, rendering, chips, screens, light, pixels, cores, speed.
4.  **Draft - Syllable Count.
Content: Cores glow bright and fast,  
Rendering light on digital screens,  
Power through silicon.


## Cleanup and shutdown

To tear down this TensorRT-LLM workflow:

1. In the terminal running `trtllm-serve`, press `Ctrl+C` to stop the server.
2. In the Docker shell, run `exit` to stop the container (`--rm` removes it automatically).
3. Optionally run the next cell to clear notebook-side CUDA cache.

In [15]:
import gc
import torch

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

print("Notebook-side CUDA cache cleanup complete.")

Notebook-side CUDA cache cleanup complete.
